In [1]:
nwb_file_name, epoch_name, brain_area = "wilbur20210403_.nwb", "08_r4", "HPC"
ripple_detector_name: str = "Kay"
non_stationary = False
training_type: str = "no_ripple"

In [2]:
import logging
import logging.config
import os
import subprocess
import warnings
from argparse import ArgumentParser, Namespace
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import dask
import networkx as nx
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client
from dask_cuda import LocalCUDACluster
from non_local_detector import NonLocalClusterlessDetector, NonLocalSortedSpikesDetector
from non_local_detector.discrete_state_transitions import DiscreteNonStationaryDiagonal
from non_local_detector.environment import Environment

from continuum_swr_replay.analysis.detection.nonlocal_events import get_data_over_time
from continuum_swr_replay.decoding.results import load_results
from continuum_swr_replay.data_loaders import (
    get_epoch_info,
    get_training_timepoints,
    load_data,
)
from continuum_swr_replay.decoding.config import nonlocal_detector_classes
from continuum_swr_replay.parameters import (
    BLOCK_SIZE,
    PLACE_BIN_SIZE_1D,
    PLACE_BIN_SIZE_2D,
    POSITION_STD,
    PROCESSED_DATA_DIR,
    WAVEFORM_STD,
)
from continuum_swr_replay.data_types import (
    ClusterlessParams,
    SessionData,
    SortedSpikesParams,
)
from continuum_swr_replay.visualization.figurl import (
    create_interactive_1D_decoding_figurl,
)

/cumulus/edeno/continuum-swr-replay/.venv/lib/python3.11/site-packages/non_local_detector/likelihoods/clusterless_gmm.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm  # type: ignore[import-untyped]
/cumulus/edeno/continuum-swr-replay/.venv/lib/python3.11/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2025-12-18 10:57:36,535][INFO]: DataJoint 0.14.6 connected to edeno@lmf-db.cin.ucsf.edu:3306


In [3]:
epoch_identifier = f"{nwb_file_name.replace('_.nwb', '')}_{epoch_name}_{brain_area}"
if ripple_detector_name != "Kay":
    epoch_identifier += f"_{ripple_detector_name}"
if non_stationary:
    epoch_identifier += "_non_stationary"

data = load_data(
    nwb_file_name,
    epoch_name,
    ripple_detector_name=ripple_detector_name,
    parallel=False,
)

[2025-12-18 11:02:49,030][WARNING]: Skipped checksum for file with hash: e5eeb507-0103-37ff-3dd9-76a9debe727b, and path: /stelmo/nwb/raw/wilbur20210403_.nwb
[2025-12-18 11:02:49,034][WARNING]: Skipped checksum for file with hash: e5eeb507-0103-37ff-3dd9-76a9debe727b, and path: /stelmo/nwb/raw/wilbur20210403_.nwb
[2025-12-18 11:02:49,037][WARNING]: Skipped checksum for file with hash: e5eeb507-0103-37ff-3dd9-76a9debe727b, and path: /stelmo/nwb/raw/wilbur20210403_.nwb
[2025-12-18 11:02:49,040][WARNING]: Skipped checksum for file with hash: e5eeb507-0103-37ff-3dd9-76a9debe727b, and path: /stelmo/nwb/raw/wilbur20210403_.nwb
[2025-12-18 11:02:49,043][WARNING]: Skipped checksum for file with hash: e5eeb507-0103-37ff-3dd9-76a9debe727b, and path: /stelmo/nwb/raw/wilbur20210403_.nwb
[2025-12-18 11:02:49,046][WARNING]: Skipped checksum for file with hash: e5eeb507-0103-37ff-3dd9-76a9debe727b, and path: /stelmo/nwb/raw/wilbur20210403_.nwb
[2025-12-18 11:02:49,056][WARNING]: Skipped checksum for f

In [4]:
position_info = data["position_info"].dropna(
    subset=["head_position_x", "head_position_y"]
)
n_dropped = len(data["position_info"]) - len(position_info)
n_dropped

0

In [6]:
data.keys()

dict_keys(['position_info', 'track_graph', 'linear_edge_order', 'linear_edge_spacing', 'spike_times', 'spike_waveform_features', 'ripple_times', 'ripple_filtered_lfps', 'ripple_lfps', 'ripple_consensus_trace', 'zscored_ripple_consensus_trace', 'theta_filtered_lfp', 'theta_phase', 'theta_power', 'beam_breaks', 'pump_events', 'light_events', 'well_visits', 'trials', 'task_variables', 'hse_times', 'multiunit_firing_rate', 'multiunit_rate_zscore', 'electrode_group_info'])

In [7]:
data["position_info"].shape

(546100, 11)

In [8]:
position_info.shape

(546100, 11)

In [9]:
data["multiunit_rate_zscore"]["HPC"].shape

(546100, 1)

In [10]:
# Create data dict with filtered position_info for get_training_timepoints
data_filtered = {**data, "position_info": position_info}
is_training = get_training_timepoints(
    data_filtered, training_type=training_type
)

# Get spike waveform features if available (HPC only)
spike_waveform_features = data["spike_waveform_features"].get(
    brain_area, None
)

In [11]:
[feat.shape for feat in spike_waveform_features]

[(60042, 4),
 (0, 4),
 (3383, 4),
 (5718, 4),
 (33951, 4),
 (0, 4),
 (13363, 4),
 (34859, 4),
 (43049, 4),
 (6815, 4),
 (3425, 4),
 (19574, 3),
 (334, 4)]

In [12]:
len(spike_waveform_features)

13

In [13]:
track_graph = data["track_graph"]
linear_edge_order = data["linear_edge_order"]
linear_edge_spacing = data["linear_edge_spacing"]

env = Environment(
    place_bin_size=PLACE_BIN_SIZE_1D,
    track_graph=track_graph,
    edge_order=linear_edge_order,
    edge_spacing=linear_edge_spacing,
)

env

Environment(environment_name='', place_bin_size=2.0, track_graph=<networkx.classes.graph.Graph object at 0x7f4ee1b937d0>, edge_order=[(9, 6), (6, 0), (6, 1), (9, 7), (7, 2), (7, 3), (9, 8), (8, 4), (8, 5)], edge_spacing=15, is_track_interior=None, position_range=None, infer_track_interior=True, fill_holes=False, dilate=False, bin_count_threshold=0, edges_=None, place_bin_edges_=None, place_bin_centers_=None, centers_shape_=None, is_track_interior_=None, is_track_boundary_=None, track_graphDD=None, distance_between_nodes_=None, track_graph_with_bin_centers_edges_=None, original_nodes_df_=None, place_bin_edges_nodes_df_=None, place_bin_centers_nodes_df_=None, nodes_df_=None, _is_fitted=False)

In [14]:
@dataclass(frozen=True)
class DecodingConfig:
    """Configuration for neural decoding."""

    position_std: float = POSITION_STD
    block_size: int = BLOCK_SIZE
    waveform_std: float = WAVEFORM_STD

    def to_clusterless_params(self) -> ClusterlessParams:
        """Get parameters for clusterless decoding (HPC)."""
        return {
            "position_std": self.position_std,
            "waveform_std": self.waveform_std,
            "block_size": self.block_size,
        }

    def to_sorted_spikes_params(self) -> SortedSpikesParams:
        """Get parameters for sorted spikes decoding (mPFC/OFC)."""
        return {
            "position_std": self.position_std,
            "block_size": self.block_size,
        }

In [15]:
# Use default stationary transition matrix (None triggers sensible defaults)
discrete_transition_covariate_data = None
discrete_transition_type = None

In [16]:
# Detector class and params type by brain area
DETECTOR_CLASSES = nonlocal_detector_classes()

def create_detector(
    brain_area: str,
    env: Environment,
    config: DecodingConfig,
    discrete_transition_type: str,
) -> NonLocalClusterlessDetector | NonLocalSortedSpikesDetector:
    """Factory for creating the appropriate detector for a brain area.

    Parameters
    ----------
    brain_area : str
        Brain area name (must be 'HPC', 'mPFC', or 'OFC')
    env : Environment
        Environment configuration
    config : DecodingConfig
        Decoding parameters
    discrete_transition_type : str
        Type of transition model

    Returns
    -------
    detector : NonLocalClusterlessDetector | NonLocalSortedSpikesDetector
        Configured detector instance

    Raises
    ------
    ValueError
        If brain_area is not recognized
    """
    if brain_area not in DETECTOR_CLASSES:
        valid_areas = list(DETECTOR_CLASSES.keys())
        raise ValueError(
            f"Invalid brain_area '{brain_area}'. Must be one of {valid_areas}"
        )

    detector_class, param_type = DETECTOR_CLASSES[brain_area]
    params = getattr(config, f"to_{param_type}_params")()

    param_key = f"{param_type}_algorithm_params"
    return detector_class(
        environments=env,
        **{param_key: params},
        discrete_transition_type=discrete_transition_type,
    )

In [17]:
config = DecodingConfig()

detector = create_detector(brain_area, env, config, discrete_transition_type)

spike_times = data["spike_times"][brain_area]

In [18]:
detector

,discrete_initial_conditions,"array([1., 0., 0., 0.])"
,continuous_initial_conditions_types,"[UniformInitialConditions(), UniformInitialConditions(), ...]"
,discrete_transition_type,DiscreteStati...0.98 ]]))
,discrete_transition_concentration,1.0
,discrete_transition_stickiness,"array([1.e+06...e+02, 3.e+02])"
,discrete_transition_regularization,1e-10
,continuous_transition_types,"[[<non_local_de...x7f4cff9f15d0>, <non_local_de...x7f4cff9f2350>, ...], [<non_local_de...x7f4cff9f1050>, <non_local_de...x7f4cff9f0350>, ...], ...]"
,observation_models,"[ObservationMo...o_spike=False), ObservationMo...no_spike=True), ...]"
,environments,"(Environment(e..._fitted=False),)"
,clusterless_algorithm,'clusterless_kde'
,clusterless_algorithm_params,"{'block_size': 1024, 'position_std': 3.5355339059327378, 'waveform_std': 24.0}"


In [19]:
# Prepare common parameters for all detectors
# position_info has already been filtered to remove NaN values upstream
estimate_params = {
    "position_time": position_info.index.to_numpy(),
    "position": position_info[["head_position_x", "head_position_y"]].to_numpy(),
    "spike_times": spike_times,
    "time": position_info.index.to_numpy(),
    "discrete_transition_covariate_data": discrete_transition_covariate_data,
}

# HPC-specific parameters: is_training and waveform features
if brain_area == "HPC":
    estimate_params["is_training"] = is_training
    if spike_waveform_features is not None:
        estimate_params["spike_waveform_features"] = spike_waveform_features

In [21]:
results = detector.estimate_parameters(**estimate_params)

W1218 18:44:55.359788 3826579 gpu_helpers.cc:80] Unable to enable peer access between GPUs 0 and 9; status: INTERNAL: failed to enable peer access from 0x7f4888c63300 to 0x7f48a8c638f0: INTERNAL: CUDA error: : CUDA_ERROR_TOO_MANY_PEERS: peer mapping resources exhausted
W1218 18:44:55.362168 3826579 gpu_helpers.cc:80] Unable to enable peer access between GPUs 1 and 9; status: INTERNAL: failed to enable peer access from 0x7f48a0c64760 to 0x7f48a8c638f0: INTERNAL: CUDA error: : CUDA_ERROR_TOO_MANY_PEERS: peer mapping resources exhausted
W1218 18:44:55.364152 3826579 gpu_helpers.cc:80] Unable to enable peer access between GPUs 2 and 9; status: INTERNAL: failed to enable peer access from 0x7f4890efba10 to 0x7f48a8c638f0: INTERNAL: CUDA error: : CUDA_ERROR_TOO_MANY_PEERS: peer mapping resources exhausted
W1218 18:44:55.365834 3826579 gpu_helpers.cc:80] Unable to enable peer access between GPUs 3 and 9; status: INTERNAL: failed to enable peer access from 0x7f48acc63720 to 0x7f48a8c638f0: INTE

Encoding models:   0%|          | 0/13 [00:00<?, ?electrode/s]

Local Likelihood:   0%|          | 0/13 [00:00<?, ?electrode/s]

No Spike Likelihood:   0%|          | 0/13 [00:00<?, ?cell/s]

Non-Local Likelihood:   0%|          | 0/13 [00:00<?, ?electrode/s]

/cumulus/edeno/continuum-swr-replay/.venv/lib/python3.11/site-packages/jax/_src/interpreters/mlir.py:1280: UserWarning: Some donated buffers were not usable: float32[490].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


... (73 outputs left out)

Local Likelihood:   0%|          | 0/13 [00:00<?, ?electrode/s]

No Spike Likelihood:   0%|          | 0/13 [00:00<?, ?cell/s]

Non-Local Likelihood:   0%|          | 0/13 [00:00<?, ?electrode/s]

Encoding models:   0%|          | 0/13 [00:00<?, ?electrode/s]

In [ ]:
results.isel(time=slice(100_000, 120_000)).acausal_state_probabilities.plot(x="time", hue="states")

In [22]:
results

<xarray.Dataset> Size: 1GB
Dimensions:                      (time: 546100, state_bins: 506, states: 4,
                                  state_ind: 506)
Coordinates:
  * time                         (time) float64 4MB 1.617e+09 ... 1.617e+09
  * state_bins                   (state_bins) object 4kB MultiIndex
  * states                       (states) <U20 320B 'Local' ... 'Non-Local Fr...
  * state_ind                    (state_ind) int64 4kB 0 1 2 2 2 2 ... 3 3 3 3 3
    environments                 (states) <U1 16B '' '' '' ''
    encoding_groups              (states) int64 32B 0 0 0 0
  * state                        (state_bins) object 4kB 'Local' ... 'Non-Loc...
  * position                     (state_bins) float64 4kB nan nan ... 596.4
Data variables:
    acausal_posterior            (time, state_bins) float32 1GB nan nan ... nan
    acausal_state_probabilities  (time, states) float32 9MB nan nan ... nan nan
Attributes:
    marginal_log_likelihoods:  [nan nan nan nan nan nan nan nan nan nan nan n...

In [ ]:
# Compute sampling frequency with validation
non_local_threshold = 0.8
time_diffs = np.diff(results.time.values)
if len(time_diffs) == 0:
    raise ValueError(
        "Time array must have at least 2 elements to compute sampling frequency"
    )
if np.any(time_diffs <= 0):
    n_bad = np.sum(time_diffs <= 0)
    bad_indices = np.where(time_diffs <= 0)[0][:5]  # First 5 problematic indices
    bad_values = time_diffs[bad_indices]
    raise ValueError(
        f"Time array must be strictly increasing with positive differences. "
        f"Found {n_bad}/{len(time_diffs)} non-positive differences. "
        f"First problematic indices: {bad_indices.tolist()}, values: {bad_values.tolist()}"
    )

median_dt = np.median(time_diffs)
if median_dt <= 1e-10:
    raise ValueError(
        f"Median time difference too small ({median_dt}), cannot compute sampling frequency"
    )

sampling_frequency = 1 / median_dt
data_over_time, posterior = get_data_over_time(
    data,
    results,
    env,
    nwb_file_name,
    epoch_name,
    brain_area,
    sampling_frequency=sampling_frequency,
    non_local_threshold=non_local_threshold,
)

In [ ]:
%debug

In [ ]:
results.time.shape